In [2]:
"""
====================================================
ERA5 Visualization Script
====================================================
"""

'\n====================================================\nERA5 Visualization Script\n====================================================\n'

In [3]:
#######################
#DIRECTORIES

In [4]:
# #SETTING UP DIRECTORIES
# mainDirectory = '/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/Regional-MPAS-Project/'
# workingDirectory="/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/Regional-MPAS-Project/DataAnalysis/InputData_DataAnalysis/"
# print(workingDirectory)
# outputDirectory=workingDirectory+"OUTPUT/"
# dataDirectory=mainDirectory+"DownloadData/DATA/ERA5_Data/"

In [5]:
#SETTING UP DIRECOTRIES
mainDirectory = "/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/"
outputDirectory=mainDirectory+"../OUTPUT/DataAnalysis/InputData_DataAnalysis/"
import os; os.makedirs(outputDirectory, exist_ok=True)
dataDirectory=mainDirectory+"../DATA/ERA5_Data/"

In [6]:
#######################
#LIBRARIES, FUNCTIONS, and CLASSES

In [7]:
#IMPORT LIBRARIES
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Libraries/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "Libraries",
]

for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [8]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Functions_2.0/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [9]:
#IMPORT CLASSES
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Functions_2.0/Classes/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "Classes_InputData_DataAnalysis",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [10]:
###########################
#FUNCTIONS

In [11]:
#MAKE DATE FOLDER (for output) FUNCTION
def MakeDateFolder(date_string):
    date_folder = strings.DateString(date_string)
    #adding date to output folder
    subdir = os.path.join(outputDirectory, date_folder)
    os.makedirs(subdir, exist_ok=True)
    return date_folder

def GetLoadDirectory(dataDirectory,date_folder):
    loadDirectorys = [
        os.path.join(dataDirectory, date_folder, f"{var}_ERA5_{date_folder}.nc")
        for var in variables.keys()
    ]
    return loadDirectorys

In [18]:
#RUN CALCULATIONS and PLOTTING #*#*#*#*#*#*# (this version adds quiver to U/V plots)
def RunCalculations(numerics, var_data, units, variable, calculation, mult_factor):
    calculation_results = {}

    arr   = var_data
    units = units
    if mult_factor != "NaN":
        arr *= mult_factor

    t, _ = Ultimate_AreaAverage(var_data, dims=('t','y','x'), dim_names=('t',), mode='keep')

    three_hours = 3 * numerics.hour_index
    tyx_3h = calculation.block_vertical_profiles_3D(arr, block=three_hours)

    calculation_results[variable] = {
        "units": units,
        "t": t,            # (t,)
        "tyx_3h": tyx_3h # (nblocks, y, x)
    }

    return calculation_results

def RunPlots(numerics, calculation_results, date_string, UTC_offset, outputFile, plotting, 
             colormap, vline, data_lim, line_contour, center_contour):
    for name, result in calculation_results.items():
        common_args = {
            "var_name": name,
            "var_units": result["units"],
            "date_string": date_string,
            "date_folder":  date_folder,
            "outputFile": outputFile,
            "numerics": numerics,
            "data_lim": data_lim,
            "UTC_offset": UTC_offset,
        }

        plotting.TimeSeries(var_data=result['t'], **common_args)
        plotting.MultiAverage_HorizontalFields_Surface(var_data=result['tyx_3h'], plev=1000, line_contour=line_contour, colormap=colormap, center_contour=center_contour, **common_args)

#RUNNING CALCULATIONS FUNCTION
def RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting):
    # global variable,calculation_results_temp1,calculation_results_temp2 #*#* quiver
    for count, (loadDirectory, (variable, components)) in enumerate(tqdm(zip(loadDirectorys, variables.items()), total=len(loadDirectorys), desc="Running Calculations"),start=1):
        units, colormap, vline, mult_factor = components["unit"], components["colormap"], components["vline"], components["mult_factor"]
        data_lim, line_contour, center_contour = components["data_lim"], components["line_contour"], components["center_contour"]
        
        #print
        print(f"Plotting {len(variables)} Variables",'\n')
        print(f"{count}. {variable} ({units}) → {loadDirectory}")
        
        #loading the variable
        ncFile=xr.open_dataset(loadDirectory)
        var_name = [v for v in list(ncFile.data_vars) if v not in ["number", "expver"]][0]
        # print('\n',var_name,'***')
        var_data=ncFile[var_name].data
        numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=None, Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,
                           TIME=ncFile[var_name]['valid_time'].data,P=[1000], LAT=ncFile[var_name]['latitude'].data,LON=ncFile[var_name]['longitude'].data)
        print(variable+":\n","\t(Nt, Nlat, Nlon) = ",(numerics.Nt,numerics.Nlat,numerics.Nlon),"\n")
    
        #making output filename
        outputFile = os.path.join(outputDirectory, date_folder, variable) #variable also can be var_name
        
        os.makedirs(outputFile, exist_ok=True)
    
        #doing calculations
        calculation_results=RunCalculations(numerics, var_data, "("+units+")", variable, calculation, mult_factor)
    
        #plotting
        RunPlots(numerics, calculation_results, date_string, UTC_offset, outputFile, plotting, 
                 colormap, vline, data_lim, line_contour, center_contour)

In [19]:
###########################
#LOADING DATA

In [20]:
#load in ERA5 data
variables = {
    "convective_available_potential_energy": {"unit": r"$J\ kg^{-1}$", "colormap": "YlOrRd", "vline": "NaN", "mult_factor": "NaN", "data_lim": "NaN", "line_contour": "F", "center_contour": "NaN",},
}

In [21]:
###########################
#RUNNING

In [22]:
#TRACER DATA
UTC_offset="-5"

In [25]:
###########################
#DATE ONE (BORING CASE)

#date information
date_string = "06-08 - 06-10 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/1 [00:00<?, ?it/s]/glade/derecho/scratch/aroseman/tmp/ipykernel_58482/1579065785.py:56: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=None, Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 1 Variables 

1. convective_available_potential_energy ($J\ kg^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/06-08_-_06-10_2022/convective_available_potential_energy_ERA5_06-08_-_06-10_2022.nc
convective_available_potential_energy:
 	(Nt, Nlat, Nlon) =  (72, 20, 23) 



Running Calculations: 100%|██████████| 1/1 [00:13<00:00, 13.30s/it]


In [26]:
###########################
#DATE TWO (RAINY CASE)

#date information
date_string = "06-30 - 07-02 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/1 [00:00<?, ?it/s]/glade/derecho/scratch/aroseman/tmp/ipykernel_58482/1579065785.py:56: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=None, Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 1 Variables 

1. convective_available_potential_energy ($J\ kg^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/06-30_-_07-02_2022/convective_available_potential_energy_ERA5_06-30_-_07-02_2022.nc
convective_available_potential_energy:
 	(Nt, Nlat, Nlon) =  (72, 20, 23) 



Running Calculations: 100%|██████████| 1/1 [00:07<00:00,  7.23s/it]


In [27]:
###########################
#DATE THREE (INTERESTING CASE)

#date information
date_string = "08-11 - 08-13 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/1 [00:00<?, ?it/s]/glade/derecho/scratch/aroseman/tmp/ipykernel_58482/1579065785.py:56: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=None, Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 1 Variables 

1. convective_available_potential_energy ($J\ kg^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/08-11_-_08-13_2022/convective_available_potential_energy_ERA5_08-11_-_08-13_2022.nc
convective_available_potential_energy:
 	(Nt, Nlat, Nlon) =  (72, 20, 23) 



Running Calculations: 100%|██████████| 1/1 [00:07<00:00,  7.09s/it]


In [70]:
######################################

In [ ]:
#PRECIP DATA (will need to make some changes to code to allow for differences in files between TRACER and PRECIP, or will conform the PRECIP nc metadata ***)
# UTC_offset=***

In [ ]:
###########################
#DATE ONE (BORING CASE)

In [ ]:
###########################
#DATE TWO (RAINY CASE)

In [ ]:
###########################
#DATE THREE (INTERESTING CASE)

In [ ]:
######################################

In [56]:
#*#* Some Possible Future Improvements #*#*
#1. All Plots: x- and y-ticks somewhat incomplete